# Deforestation change detection: project walkthrough

This notebook explains the project from start to finish using the code in `src/` and the results
already saved on disk. It reuses the project's own functions rather than repeating them.

**How to use it**
1. In VS Code, click **Select Kernel** and choose `.venv` (`.venv\Scripts\python.exe`).
2. Run the cells from top to bottom.
3. Compare your result with the `Expected output` comment at the end of each code cell.

**It is safe to run.** No cell contacts Earth Engine, spends EECU, trains a model, or changes your data.

**Where the project stands:** Phase 0 is done. Phase 1 has run on the NE quarter, and the other
three quarters are exporting. Phases 2 to 5 have not started, so no accuracy results exist yet.

## The big picture

```
Sentinel-2 (optical)  ─┐
Sentinel-1 (radar)    ─┼─> Earth Engine ─> GeoTIFFs ─> patches ─> model ─> loss map + hectares
Hansen (labels)       ─┘   composites      data/raw    data/patches
```

**The goal:** compare a place in 2020 with the same place in 2022, and find where forest was lost.

**Why two satellites?** Sentinel-2 is a camera, so clouds block it. Sentinel-1 is radar, which sends
its own signal and passes through cloud.

**Why not a simple greenness threshold?** Greenness finds low vegetation, not forest that was lost.
This project learns from Hansen Global Forest Change, a published map of yearly tree-cover loss.

## Step 1: Setup

This cell finds the project folder and imports the functions used in the rest of the notebook.
`config` holds all the settings, so no values are hard-coded here.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from src.config import load_config
from src.data import gee
from src.data.labels import IGNORE, LOSS, NO_LOSS, derive_label
from src.data.patches import image_band_indices, read_cell, source_prefixes

config = load_config()
AREA = "subset"  # the NE quarter; becomes "full" once every quarter is processed

print("Project folder found:", (ROOT / "configs" / "default.yaml").exists())

# Expected output:
# Project folder found: True

**Next step:** look at the settings that control the whole pipeline.

## Step 2: The configuration

`configs/default.yaml` holds the region, years, season, patch size and paths. `src/config.py` loads
it and stops with a clear error if a value is invalid, so a typo can't waste an export.

- **Region:** São Félix do Xingu, Pará, Brazil, a deforestation hotspot.
- **Season:** June to September, the dry season, when clouds are fewest and both years are comparable.
- **Subset:** the NE quarter, used to test everything cheaply before committing to the full box.

In [ ]:
box = config.region.bbox
print(f"Region:  {config.region.name}")
print(f"Box:     west {box.west}, south {box.south}, east {box.east}, north {box.north}")
print(f"Subset:  {config.subset.name}")
print(f"Years:   {config.years.before} to {config.years.after}")
print(f"Season:  months {config.season.start_month} to {config.season.end_month}")
print(f"Pixels:  {config.export.scale_m} m in {config.export.crs}")
print(f"Patches: {config.patch.size} x {config.patch.size} pixels")
print(f"Forest:  tree cover of at least {config.labels.forest_min_treecover}%")

# Expected output:
# Region:  sao_felix_do_xingu
# Box:     west -52.5, south -7.0, east -51.5, north -6.0
# Subset:  ne_quarter
# Years:   2020 to 2022
# Season:  months 6 to 9
# Pixels:  10 m in EPSG:32722
# Patches: 256 x 256 pixels
# Forest:  tree cover of at least 50%

**Next step:** confirm the computer can train models.

## Step 3: Check the GPU (Phase 0)

Training needs the GPU. The real check is `python -m scripts.check_env`, which also runs one
training step. This cell is the short version.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Expected output:
# PyTorch: 2.14.1+cu130
# CUDA available: True
# GPU: NVIDIA GeForce RTX 4060 Laptop GPU

**Next step:** check whether the radar satellite covers this region well enough in both years.

## Step 4: Is there enough radar data?

A composite merges many images of a place into one by taking the median. Radar images are grainy
(this noise is called speckle), so more images give a cleaner composite.

`scripts/check_s1_coverage.py` counted Sentinel-1 images per pixel on a coarse 1 km grid and saved
the result. This cell reads that file, so it costs nothing.

In [ ]:
coverage = json.loads((config.paths.outputs_dir / "phase1" / "s1_coverage.json").read_text())

for year, result in coverage["years"].items():
    season = result["season"]
    print(f"{year}: images per pixel min {season['min']}, median {season['median']}, max {season['max']}; "
          f"pixels under 3 images: {season['share_under_3']:.0%}")

# Expected output:
# 2020: images per pixel min 9, median 9, max 18; pixels under 3 images: 0%
# 2022: images per pixel min 6, median 7, max 7; pixels under 3 images: 0%

**What this means:** every pixel has at least 6 images, which is enough for a median. All images
come from one orbit track, so both years view the ground from the same angle. 2022 has fewer images,
with a 48-day gap from June 30 to August 17, so its composite will be slightly noisier.

**Next step:** see what goes into each exported image.

## Step 5: What is in the exported image?

`gee.build_stack` in `src/data/gee.py` tells Earth Engine how to build one image with 28 layers
(bands). For each year:

- **Sentinel-2, 10 bands:** cloudy pixels are removed, then the median of what's left is taken.
- **Sentinel-1, 2 bands:** VV and VH are the two radar polarisations, also as a median.
- **Clear count, 1 band:** how many cloud-free Sentinel-2 images went into that pixel.

Then 2 Hansen bands are added: `lossyear` and `treecover2000`. They stay raw so label rules can
change later without spending EECU on a new export.

In [ ]:
band_names = gee.stack_band_names(config)
model_bands = [band_names[i] for i in image_band_indices(band_names)]

print("Total bands:", len(band_names))
print("Bands for 2020:", band_names[:13])
print("Hansen bands:", band_names[-2:])
print("Bands the model will see:", len(model_bands))

# Expected output:
# Total bands: 28
# Bands for 2020: ['B2_2020', 'B3_2020', 'B4_2020', 'B5_2020', 'B6_2020', 'B7_2020', 'B8_2020', 'B8A_2020', 'B11_2020', 'B12_2020', 'VV_2020', 'VH_2020', 's2_clear_count_2020']
# Hansen bands: ['lossyear', 'treecover2000']
# Bands the model will see: 24

**Next step:** turn the raw Hansen bands into the answers the model learns from.

## Step 6: How labels are made

This rule is the most important design decision in the project. Hansen says which *calendar year*
a pixel was lost, but the composites only cover June to September:

```
2020 composite                      2022 composite
   Jun-Sep    Oct 2020 ... May 2022    Jun-Sep
```

- **Loss in 2021** is certainly between the two composites, so it's labeled **loss (1)**.
- **Loss in 2020 or 2022** could fall on either side of a composite, so it's **ignored (255)**.
  The model is never scored on these pixels.
- **Loss before 2020 or after 2022** doesn't show as a change between the composites, so it's **no loss (0)**.
- **2021 loss below 50% tree cover** isn't clearly forest, so it's **ignored (255)**.

Here is `derive_label` on six example pixels. Hansen stores years as years since 2000, so 21 means 2021.

In [ ]:
lossyear = np.array([[21, 21, 20, 22, 19, 0]], dtype=np.int16)
treecover = np.array([[90, 40, 90, 90, 90, 90]], dtype=np.int16)

label = derive_label(lossyear, treecover, config.years.before, config.years.after,
                     config.labels.forest_min_treecover, gee.NODATA)

examples = pd.DataFrame({
    "lossyear": lossyear[0],
    "treecover": treecover[0],
    "label": label[0],
    "meaning": [{NO_LOSS: "no loss", LOSS: "loss", IGNORE: "ignored"}[v] for v in label[0]],
})
print(examples.to_string(index=False))

# Expected output:
#  lossyear  treecover  label meaning
#        21         90      1    loss
#        21         40    255 ignored
#        20         90    255 ignored
#        22         90    255 ignored
#        19         90      0 no loss
#         0         90      0 no loss

**Next step:** measure what this rule keeps and drops across the whole box.

## Step 7: What the label rule costs

`scripts/report_labels.py` counted every Hansen pixel in the box by tree cover and loss year. This
shows what each choice keeps. Counts are in Hansen's 30 m pixels.

In [ ]:
labels = json.loads((config.paths.outputs_dir / "phase1" / "label_report.json").read_text())

rows = {}
for threshold in (30, 50, 75):
    result = labels["FULL"][f"treecover_ge_{threshold}"]
    rows[f"cover >= {threshold}%"] = {
        "2021 loss kept": result["loss_middle_years_kept"],
        "2020 ignored": result["loss_2020_ignored"],
        "2022 ignored": result["loss_2022_ignored"],
        "hectares kept": round(result["kept_hectares_approx"]),
    }
print(pd.DataFrame(rows).T.to_string())

print("\n2021 loss kept per quarter at 50% cover:")
for quarter in ("NW", "NE", "SW", "SE"):
    print(f"  {quarter}: {labels[quarter]['treecover_ge_50']['loss_middle_years_kept']:,}")

# Expected output:
#               2021 loss kept  2020 ignored  2022 ignored  hectares kept
# cover >= 30%          165270        337719        176047          12718
# cover >= 50%          158436        327692        169613          12192
# cover >= 75%          141511        306022        156811          10890
#
# 2021 loss kept per quarter at 50% cover:
#   NW: 40,186
#   NE: 72,154
#   SW: 28,495
#   SE: 17,601

**What this means:** only about a quarter of 2020 to 2022 loss is kept, but those labels can be
trusted. The 50% threshold loses only about 4% of positives compared with 30%. The NE quarter has
the most loss, which is why it was chosen for the test.

**Next step:** see how the imagery got from Earth Engine to this computer.

## Step 8: Exporting the imagery

These are the commands that ran. **Don't run them here**, because they spend real quota. The script
needs `--confirm`, and it refuses to repeat an export that already completed.

```powershell
python -m scripts.export_stack --area probe                    # measures EECU on a tiny area
python -m scripts.export_stack --area subset --confirm --wait  # NE quarter
python -m scripts.export_stack --area nw --confirm --wait      # also sw and se
```

| Export | Area | EECU-hours used | Time |
|---|---|---|---|
| Probe | 26 km² | 0.02 | 1 min |
| NE quarter | ~3,060 km² | 4.3 | 17.5 min |
| NW quarter | ~3,060 km² | 4.2 | 20 min |

Each quarter arrives as 9 GeoTIFF files, split at 2,560 pixels. This cell opens one file. Rasterio
reads only the header, so it's instant.

In [ ]:
prefixes = source_prefixes(config, AREA)
manifest = json.loads((config.paths.raw_dir / f"{prefixes[0]}.manifest.json").read_text())
raw_files = sorted(p for prefix in prefixes for p in config.paths.raw_dir.glob(f"{prefix}*.tif")
                   if not p.name.endswith(".local.tif"))

print("Files for this area:", len(raw_files))
with rasterio.open(raw_files[0]) as source:
    print("Size:", source.width, "x", source.height, "pixels")
    print("Bands:", source.count)
    print("Data type:", source.dtypes[0])
    print("Projection:", source.crs)
    print("First band names:", list(source.descriptions[:3]))
    print("All band names match the manifest:", list(source.descriptions) == manifest["bands"])

# Expected output:
# Files for this area: 9
# Size: 2560 x 2560 pixels
# Bands: 28
# Data type: int16
# Projection: EPSG:32722
# First band names: ['B2_2020', 'B3_2020', 'B4_2020']
# All band names match the manifest: True

The manifest is saved next to the files and records the band order, nodata value and radar scaling.
Radar values are stored as dB × 100, so `-1054` means -10.54 dB.

**Next step:** cut these large files into pieces a model can use.

## Step 9: Cutting patches

`python -m scripts.make_patches --area subset` turns the large files into 256 × 256 pixel patches.
This step is local and free, and you can repeat it whenever you like.

All exports share one map grid, so patches from different files can't partly overlap. A patch that
crosses a file boundary is rebuilt by `read_cell`.

To see that nothing is hidden, this cell rebuilds the patch with the most loss from the raw files,
then checks it matches the saved one exactly.

In [ ]:
patch_dir = config.paths.patches_dir / AREA
metadata = pd.read_csv(patch_dir / "metadata.csv")
print("Patches:", len(metadata))

record = metadata.sort_values("loss_pixels", ascending=False).iloc[0]
print("Most loss:", record.patch_id, "with", record.loss_pixels, "loss pixels, split:", record.split)

sources = [rasterio.open(path) for path in raw_files]
cell = read_cell(sources, int(record.left), int(record.top), config.patch.size,
                 config.export.scale_m, len(manifest["bands"]), manifest["nodata"])
for source in sources:
    source.close()

saved = np.load(patch_dir / f"{record.patch_id}.npz")
print("Rebuilt from raw:", cell.shape)
print("Saved image:", saved["image"].shape, "| label:", saved["label"].shape,
      "| clear count:", saved["clear_count"].shape)
print("Images match exactly:", np.array_equal(cell[image_band_indices(manifest["bands"])], saved["image"]))

# Expected output:
# Patches: 420
# Most loss: n9295360_e417280 with 33958 loss pixels, split: val
# Rebuilt from raw: (28, 256, 256)
# Saved image: (24, 256, 256) | label: (256, 256) | clear count: (2, 256, 256)
# Images match exactly: True

Each patch has 24 input bands, 1 label layer, and 2 quality layers that the model won't use.

**Next step:** look at a patch.

## Step 10: Looking at a patch

Satellite bands aren't normal photo colors. This uses **false color**: shortwave infrared, near
infrared and red. Forest appears green, and bare ground appears brown or pink.

The radar panel shows how VH changed. Red means the signal dropped, which usually happens when tree
cover is removed.

In [ ]:
image, label = saved["image"], saved["label"]
names = [band_names[i] for i in image_band_indices(band_names)]

def false_color(year):
    bands = [image[names.index(f"{band}_{year}")] for band in ("B11", "B8", "B4")]
    return np.clip(np.dstack(bands) / 4000, 0, 1)

vh_change = (image[names.index("VH_2022")].astype(float) - image[names.index("VH_2020")]) / manifest["s1_db_scale"]

figure, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(false_color(2020)); axes[0].set_title("2020")
axes[1].imshow(false_color(2022)); axes[1].set_title("2022")
axes[2].imshow(vh_change, cmap="RdBu", vmin=-6, vmax=6); axes[2].set_title("VH change (dB)")
axes[3].imshow(np.where(label == IGNORE, 0.5, label), cmap="gray"); axes[3].set_title("Label")
for axis in axes:
    axis.axis("off")
plt.show()

print(f"Loss pixels: {(label == LOSS).sum():,}")
print(f"Ignored pixels: {(label == IGNORE).sum():,}")
print(f"No-loss pixels: {(label == NO_LOSS).sum():,}")

# Expected output:
# Loss pixels: 33,958
# Ignored pixels: 12,064
# No-loss pixels: 19,514
# (plus the figure described below)

In the figure, you should see forest in 2020 replaced by bare or brown ground in 2022, with red over
the same area in the radar panel. In the label panel, white is loss, gray is ignored, and black is no
loss. Its edges look blocky because Hansen's pixels are 30 m and these are 10 m.

More samples are in `outputs/phase1/samples_subset.png`, made by `python -m scripts.view_patches --area subset`.

**Next step:** divide the patches so the final score can be trusted.

## Step 11: Train, validation and test splits

Random splits would be misleading. Neighbouring patches look alike, so a model could score well just
by remembering a place. Instead, the area is cut into north-to-south stripes:

- **Train:** the model learns from these patches.
- **Val:** used during training to choose the best version of the model.
- **Test:** used only at the end, for the final score.
- **Buffer:** dropped rows between stripes, so splits never sit side by side.

Stripes are placed by share of **loss pixels**, not area, so test can't end up with almost no loss.

In [ ]:
summary = json.loads((patch_dir / "summary.json").read_text())

table = pd.DataFrame(summary["splits"]).T[["patches", "patches_with_loss", "loss_pixels"]].astype(int)
table["loss_share_of_labeled"] = [f"{value['loss_share_of_labeled_pixels']:.1%}" for value in summary["splits"].values()]
print(table.to_string())

figure, axis = plt.subplots(figsize=(5, 5))
for split, group in metadata.groupby("split"):
    axis.scatter(group.left, group.top, s=12, label=split, marker="s")
axis.set_title("Patch locations by split")
axis.legend()
axis.set_aspect("equal")
axis.tick_params(labelbottom=False, labelleft=False)
plt.show()

# Expected output:
#         patches  patches_with_loss  loss_pixels loss_share_of_labeled
# train       300                220       298101                  1.6%
# val          20                 14        59779                  4.7%
# test         60                 57        53553                  1.4%
# buffer       40                 34       109493                  4.5%
# (plus the figure described below)

In the figure, you should see horizontal stripes: train in the north, then val and test further
south, with a buffer row between each.

**A known weakness:** this quarter's val set is thin and has about three times train's loss share.
The full box should fix that, and its numbers will replace these once patches are cut.

**Next step:** see why this class balance shapes the training plan.

## Step 12: Why accuracy would be misleading

Loss is rare. A useless model that predicts "no loss" everywhere would still score very well on plain
accuracy, as this cell shows. That's why the project only reports IoU, F1, precision and recall on
the loss class.

In [ ]:
train = summary["splits"]["train"]
labeled = train["total_pixels"] - train["ignored_pixels"]

print(f"Labeled training pixels: {labeled:,}")
print(f"Loss pixels: {train['loss_pixels']:,}")
print(f"Accuracy of always predicting 'no loss': {1 - train['loss_pixels'] / labeled:.1%}")
print("Loss pixels that model finds: 0")

# Expected output:
# Labeled training pixels: 18,920,213
# Loss pixels: 298,101
# Accuracy of always predicting 'no loss': 98.4%
# Loss pixels that model finds: 0

Phase 2 handles this with a loss function suited to rare classes (Dice plus focal) and by showing
the model loss-containing patches more often. Every metric must skip label `255`.

**Next step:** finish Phase 1, then start modeling.

## Step 13: What happens next

**Finish Phase 1**
1. The NW, SW and SE exports finish.
2. You download their `.tif` files from Drive into `data\raw\`.
3. `python -m scripts.make_patches --area full` cuts every quarter onto the shared grid.
4. We review the final split numbers, then you approve Phase 2.

**Phase 2: Baseline model**
A PyTorch dataset loads patches lazily from disk, with flips and rotations. Both dates are stacked
into a 24-channel U-Net with a pretrained encoder, trained with Dice plus focal loss and mixed
precision. The best version is chosen by validation F1 on the loss class.

**Phase 3: Evaluation**
Tune the decision threshold on val, score on test, remove specks under about 1 hectare, and build a
gallery of failures such as clouds, rivers and seasonal change.

**Phase 4: Better models**
Compare a Siamese network, and optionally SegFormer and Prithvi-EO, on exactly the same split.

**Phase 5: Demo**
Run tiled inference over large areas, calculate hectares lost, and build an app where someone picks
an area and two years, then sees the loss map.

Set `AREA = "full"` in Step 1 once full patches exist, and this notebook will show the final numbers.